In [1]:
from pathlib import Path
import pandas as pd
import json
import os
import re

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

#### MAOMAO v1.0 — Getting started

This notebook is the **minimal entry point for direct reuse of the public MAOMAO release**.

Unlike the resource-construction notebooks, this workflow does not rebuild source-specific datasets, reapply the ontology, regenerate numerical representations, or recreate benchmark partitions.

It demonstrates how to:

- locate and validate the MAOMAO release;
- load the sequence-level master resource;
- inspect the endpoint ontology and evidence-state encoding;
- inspect audit and ambiguity-support assets;
- query peptides by endpoint and evidence state;
- retrieve one peptide and follow its supporting resource-level records;
- discover released numerical representations;
- discover endpoint-specific train/validation/test partitions;
- connect a fixed split to a numerical representation through the stable MAOMAO `id`;
- optionally export a user-defined subset.

The goal is to show that the frozen MAOMAO assets can be consumed directly by an external user.

> **Scope.** This notebook demonstrates resource access and interoperability. Descriptive EDA, statistical comparisons, classification, and regression are kept in separate worked examples.


##### 0. Download the code and public data release

Clone the repository:

```bash
git clone https://github.com/kren-ai-lab/maomao.git
cd maomao
```

The complete MAOMAO v1.0.0 data release is archived on Zenodo:

**DOI: 10.5281/zenodo.22312156**

The release is organized into conceptual layers such as the Core, Embedding, Benchmark, Descriptor, and Documentation layers.

For the Core Layer, the repository documentation uses:

```bash
mkdir -p downloads/maomao_release

curl -fL   "https://zenodo.org/api/records/22312156/files/core_layer.zip/content"   -o downloads/core_layer.zip

unzip downloads/core_layer.zip -d downloads/maomao_release
```

The repository README explains how to restore the distributed layers to their operational locations.

For portable notebook execution, define either the repository root or the extracted release root:

```bash
export MAOMAO_ROOT=/path/to/maomao
```

or

```bash
export MAOMAO_ROOT=/path/to/maomao_release
```


##### 1. Locate the MAOMAO release and configure outputs

The notebook supports both:

1. the **operational repository structure**, where `processed_data/`, `numerical_representation_data/`, and `split_process/` are restored at the repository root; and
2. the **Zenodo distribution structure**, where the same assets can remain inside `core_layer/`, `embedding_layer/`, and `benchmark_layer/`.

An explicit `MAOMAO_ROOT_OVERRIDE` can be used when the environment variable is not convenient. Any optional export is written beneath `results_how_to_use_maomao/` at the resolved MAOMAO root; `OUTPUT_ROOT_OVERRIDE` can redirect it when needed.

In [2]:
MAOMAO_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None
RESULTS_DIRECTORY_NAME = "results_how_to_use_maomao"

PIVOT_RELATIVE_PATHS = [
    Path("processed_data/processed_data/maomao_sequence_pivot.csv"),
    Path("core_layer/processed_data/processed_data/maomao_sequence_pivot.csv"),
]

def candidate_roots(override=None):
    candidates = []

    if override is not None:
        candidates.append(Path(override).expanduser())

    env_root = os.environ.get("MAOMAO_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd()
    candidates.extend([cwd, *list(cwd.parents)[:4]])

    # Common location when notebooks are executed from a cloned repository.
    candidates.extend([
        cwd / "downloads" / "maomao_release",
        cwd.parent / "downloads" / "maomao_release",
    ])

    unique = []
    seen = set()
    for path in candidates:
        try:
            resolved = path.resolve()
        except FileNotFoundError:
            resolved = path.absolute()

        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)

    return unique

def resolve_maomao_root(override=None):
    checked = []

    for root in candidate_roots(override):
        for rel in PIVOT_RELATIVE_PATHS:
            candidate = root / rel
            checked.append(candidate)

            if candidate.is_file():
                return root, candidate

    preview = "\n".join(f" - {p}" for p in checked[:25])
    raise FileNotFoundError(
        "Could not locate maomao_sequence_pivot.csv.\n"
        "Set MAOMAO_ROOT or MAOMAO_ROOT_OVERRIDE to the repository "
        "or extracted Zenodo release root.\n\n"
        f"Checked examples:\n{preview}"
    )

def resolve_output_root(maomao_root):
    if OUTPUT_ROOT_OVERRIDE is not None:
        return Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve()
    return Path(maomao_root).resolve() / RESULTS_DIRECTORY_NAME

MAOMAO_ROOT, PIVOT_PATH = resolve_maomao_root(MAOMAO_ROOT_OVERRIDE)
OUTPUT_ROOT = resolve_output_root(MAOMAO_ROOT)

print("MAOMAO root :", MAOMAO_ROOT)
print("Master pivot:", PIVOT_PATH)
print("Output root :", OUTPUT_ROOT)

MAOMAO root : /home/nicole/Descargas/maomao
Master pivot: /home/nicole/Descargas/maomao/processed_data/processed_data/maomao_sequence_pivot.csv
Output root : /home/nicole/Descargas/maomao/results_how_to_use_maomao


##### 2. Validate the principal public-release assets


In [3]:
MASTER_DIR = PIVOT_PATH.parent
PROCESSED_ROOT = MASTER_DIR.parent

CORE_ASSETS = {
    "Sequence-level pivot": MASTER_DIR / "maomao_sequence_pivot.csv",
    "Resource metadata": MASTER_DIR / "metadata.json",
    "Ambiguous support": MASTER_DIR / "maomao_ambiguous_support.csv",
    "Endpoint-count audit": MASTER_DIR / "audit_endpoint_counts.csv",
    "Hierarchy-change audit": MASTER_DIR / "audit_hierarchy_changes.csv",
}

asset_status = pd.DataFrame([
    {
        "asset": name,
        "available": path.is_file(),
        "path": str(path),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.is_file() else None,
    }
    for name, path in CORE_ASSETS.items()
])

display(asset_status)

if not CORE_ASSETS["Sequence-level pivot"].is_file():
    raise FileNotFoundError("The principal MAOMAO sequence-level pivot is missing.")


,asset,available,path,size_mb
0,Sequence-level pivot,True,/home/nicole/Descargas/maomao/processed_data/processed_data/maomao_sequence_pivot.csv,8.844
1,Resource metadata,True,/home/nicole/Descargas/maomao/processed_data/processed_data/metadata.json,0.119
2,Ambiguous support,True,/home/nicole/Descargas/maomao/processed_data/processed_data/maomao_ambiguous_support.csv,1.527
3,Endpoint-count audit,True,/home/nicole/Descargas/maomao/processed_data/processed_data/audit_endpoint_counts.csv,0.000
4,Hierarchy-change audit,True,/home/nicole/Descargas/maomao/processed_data/processed_data/audit_hierarchy_changes.csv,2.626


##### 3. Load and validate the sequence-level master resource

The MAOMAO pivot contains one row per unique normalized peptide sequence.

The stable `id` is sequence-derived and is reused across the master resource, numerical representations, endpoint-specific binary datasets, and generated splits.


In [4]:
maomao = pd.read_csv(PIVOT_PATH)

ENDPOINTS = [
    "toxic",
    "cytotoxic",
    "hemolytic",
    "cytolysis",
    "neurotoxic",
    "embryotoxic",
    "ichthyotoxic",
]

required_columns = {"id", "sequence", *ENDPOINTS}
missing = sorted(required_columns - set(maomao.columns))

if missing:
    raise KeyError(f"Required MAOMAO columns are missing: {missing}")

if maomao["id"].isna().any():
    raise ValueError("Missing MAOMAO identifiers detected.")

if maomao["id"].duplicated().any():
    raise ValueError("Duplicate MAOMAO identifiers detected.")

if maomao["sequence"].isna().any():
    raise ValueError("Missing peptide sequences detected.")

maomao["sequence_length"] = maomao["sequence"].astype(str).str.len()

print(f"Rows          : {len(maomao):,}")
print(f"Unique IDs    : {maomao['id'].nunique():,}")
print(f"Columns       : {maomao.shape[1]:,}")
print(f"Length range  : {maomao['sequence_length'].min()}–{maomao['sequence_length'].max()} aa")

display(maomao.head(10))


Rows          : 71,857
Unique IDs    : 71,857
Columns       : 11
Length range  : 5–70 aa


,id,sequence,toxic,cytotoxic,hemolytic,cytolysis,anti_mammalian_cells,neurotoxic,embryotoxic,ichthyotoxic,sequence_length
0,sha256_c70d828a62440a19acb4c85c0d1cff29a5b340ac5ba671e7d5869d2e754783db,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,2,0,2,999,0,999,999,999,33
1,sha256_cb0e712239fad2596dfffa24ab580259aba066d734c3f6a28de6ac458fd9b56c,AAAAAAAAAGETS,999,0,0,999,0,999,999,999,13
2,sha256_966434f9646239a8cd0a9375d1036fb4328d803599893cbe3ec2ef6a58bd3500,AAAAAAAAAK,999,999,0,999,999,999,999,999,10
3,sha256_1de8b390ef5a86661bf0da4babf5a6059743fb66bfc8a49d168d7b1cc99d9252,AAAAAAAIKMLMDLVNERIMALNKKAKK,0,0,0,999,0,999,999,999,28
4,sha256_42c3b829be38387546e22839e4d70387e2af55cc2709863759ae1a0f965608aa,AAAAARRRIRKQAHAHSK,0,0,0,999,0,999,999,999,18
5,sha256_d28a4510f4c0205ff4f9d7876d72ccbd1a53923c83f37da04c482dbd3c2316ee,AAAADTIVLPYNDIDA,0,999,999,999,999,999,999,999,16
6,sha256_8f0f6b0173fac930e041f1f05ce1a137b1ebe1a9d5f110c4a98495e6c7ea345d,AAAAGDSAASDLLGDNILRSEDPPMSIDLTFHMLRNMIHMAKMEGEREQAQINRNLLDEV,0,999,999,999,999,999,999,999,60
7,sha256_971d4d9249953f89c0d94e3df0c5989c61b8b91a03bdba9ecaf39629aaaba925,AAAAGFEKGIDRDFEPVLFMTPLN,0,999,999,999,999,999,999,999,24
8,sha256_9a1bb3e3ce4ceecb409e3802c3ae77cd81e14f7c965a586671b63930ed9d5ddc,AAAAGFEKGIDRDFEPVLSMTPLN,0,999,999,999,999,999,999,999,24
9,sha256_339c8a3cc668390a015e03713838538b3d0b9cb06f5f83f2ee2a48ce4214ea55,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCET,0,0,2,999,0,999,999,999,44


##### Stable identifier contract


In [5]:
id_contract = pd.DataFrame({
    "id": maomao["id"].head(5),
    "sequence": maomao["sequence"].head(5),
    "sequence_length": maomao["sequence_length"].head(5),
})

display(id_contract)

valid_sha256_prefix = maomao["id"].astype(str).str.match(r"^sha256_[0-9a-f]{64}$")
print(
    "IDs matching the documented sha256_<digest> format:",
    f"{valid_sha256_prefix.sum():,}/{len(maomao):,}",
)


,id,sequence,sequence_length
0,sha256_c70d828a62440a19acb4c85c0d1cff29a5b340ac5ba671e7d5869d2e754783db,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,33
1,sha256_cb0e712239fad2596dfffa24ab580259aba066d734c3f6a28de6ac458fd9b56c,AAAAAAAAAGETS,13
2,sha256_966434f9646239a8cd0a9375d1036fb4328d803599893cbe3ec2ef6a58bd3500,AAAAAAAAAK,10
3,sha256_1de8b390ef5a86661bf0da4babf5a6059743fb66bfc8a49d168d7b1cc99d9252,AAAAAAAIKMLMDLVNERIMALNKKAKK,28
4,sha256_42c3b829be38387546e22839e4d70387e2af55cc2709863759ae1a0f965608aa,AAAAARRRIRKQAHAHSK,18


IDs matching the documented sha256_<digest> format: 71,857/71,857


##### 4. Inspect the endpoint ontology and evidence model

MAOMAO organizes the released toxicity endpoints using the hierarchy:

```text
Toxic
├── Cytotoxic
│   ├── Hemolytic
│   └── Cytolysis
├── Neurotoxic
├── Embryotoxic
└── Ichthyotoxic
```

The sequence-level pivot preserves five final evidence states:

| Code | State | Interpretation |
|---:|---|---|
| `0` | Negative | Explicit negative evidence |
| `1` | Positive | Explicit or hierarchy-supported positive evidence |
| `2` | Ambiguous | Conflicting or unresolved evidence |
| `3` | Unlabeled | Sequence was present but the endpoint was not labeled |
| `999` | No information | No usable information for the endpoint |


In [6]:
STATE_LABELS = {
    0: "Negative",
    1: "Positive",
    2: "Ambiguous",
    3: "Unlabeled",
    999: "No information",
}

state_summary = pd.DataFrame([
    {
        "endpoint": endpoint,
        **{
            STATE_LABELS[state]: int((maomao[endpoint] == state).sum())
            for state in STATE_LABELS
        },
    }
    for endpoint in ENDPOINTS
])

display(state_summary)


,endpoint,Negative,Positive,Ambiguous,Unlabeled,No information
0,toxic,35487,13824,7769,0,14777
1,cytotoxic,17238,7122,1094,0,46403
2,hemolytic,19350,4714,6852,5044,35897
3,cytolysis,2,341,0,0,71514
4,neurotoxic,1700,1481,2,0,68674
5,embryotoxic,0,2,0,0,71855
6,ichthyotoxic,0,5,0,0,71852


##### Resource metadata


In [7]:
metadata_path = CORE_ASSETS["Resource metadata"]

if metadata_path.is_file():
    with metadata_path.open("r", encoding="utf-8") as handle:
        metadata = json.load(handle)

    print("Metadata file:", metadata_path)
    print("Top-level keys:")
    for key in metadata.keys():
        print(" -", key)
else:
    metadata = None
    print("metadata.json is not available in the resolved Core Layer.")


Metadata file: /home/nicole/Descargas/maomao/processed_data/processed_data/metadata.json
Top-level keys:
 - resource
 - dataset
 - processing_configuration
 - controlled_vocabulary_and_hierarchy
 - hierarchy_application
 - input_provenance
 - annotation_before_hierarchy
 - annotation_after_hierarchy
 - ambiguity
 - sequence_statistics
 - output_files
 - metadata_record


In [8]:
def compact_json_preview(obj, max_chars=4000):
    text = json.dumps(obj, indent=2, ensure_ascii=False)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n... [preview truncated]"

if metadata is not None:
    candidate_sections = [
        "ontology",
        "hierarchy",
        "endpoints",
        "evidence",
        "statistics",
        "provenance",
        "processing_rules",
    ]

    available_sections = [key for key in candidate_sections if key in metadata]

    if available_sections:
        for key in available_sections:
            print(f"\n### metadata[{key!r}]")
            print(compact_json_preview(metadata[key], max_chars=2500))
    else:
        print(compact_json_preview(metadata, max_chars=4000))


{
  "resource": {
    "name": "MAOMAO",
    "expanded_name": "Metadata-Aware Ontology for Multi-source Annotation Organization",
    "title": "MAOMAO: An Ontology-Guided FAIR Resource for Harmonized Peptide Toxicity Data",
    "version": "1.1.0",
    "generated_at": "2026-09-05T01:12:01.780586+00:00",
    "description": "Ontology-guided, provenance-aware, and uncertainty-aware resource integrating multi-source peptide toxicity annotations.",
    "repository": "https://github.com/kren-ai-lab/maomao",
    "license": "See repository LICENSE.txt",
    "resource_type": "harmonized biological data resource"
  },
  "dataset": {
    "name": "maomao_sequence_pivot",
    "filename": "maomao_sequence_pivot.csv",
    "identifier_column": "id",
    "identifier_format": "sha256_<64_lowercase_hex>",
    "identifier_algorithm": "SHA-256",
    "identifier_input": "UTF-8 encoding of the normalized uppercase peptide sequence with whitespace removed",
    "sequence_column": "sequence",
    "annotation_col

##### 5. Inspect audit and ambiguity-support tables

MAOMAO does not expose only the final pivot. The release also retains explicit audit products that allow users to inspect how endpoint assignments were summarized and how hierarchy-based changes were recorded.


In [9]:
def read_optional_csv(path):
    if path.is_file():
        return pd.read_csv(path)
    return None

endpoint_audit = read_optional_csv(CORE_ASSETS["Endpoint-count audit"])
hierarchy_audit = read_optional_csv(CORE_ASSETS["Hierarchy-change audit"])
ambiguous_support = read_optional_csv(CORE_ASSETS["Ambiguous support"])

print("Endpoint audit :", None if endpoint_audit is None else endpoint_audit.shape)
print("Hierarchy audit:", None if hierarchy_audit is None else hierarchy_audit.shape)
print("Ambiguous data :", None if ambiguous_support is None else ambiguous_support.shape)


Endpoint audit : (8, 5)
Hierarchy audit: (17166, 10)
Ambiguous data : (13177, 7)


In [10]:
if endpoint_audit is not None:
    print("Endpoint-level audit")
    display(endpoint_audit.head(30))
else:
    print("audit_endpoint_counts.csv is not available.")


Endpoint-level audit


,toxicity_endpoint,positive,negative,ambiguous,unlabeled
0,anti_mammalian_cells,3664,17729,0,0
1,cytolysis,341,2,0,0
2,cytotoxic,7122,17238,1094,0
3,embryotoxic,2,0,0,0
4,hemolytic,4714,19350,6852,5044
5,ichthyotoxic,5,0,0,0
6,neurotoxic,1481,1700,2,0
7,toxic,13824,35487,7769,0


In [11]:
if hierarchy_audit is not None:
    print("Hierarchy-change audit")
    display(hierarchy_audit.head(20))
else:
    print("audit_hierarchy_changes.csv is not available.")


Hierarchy-change audit


,id,sequence,endpoint,status_before_hierarchy,status,hierarchy_source,is_hierarchy_inferred,hierarchy_blocked_by_ambiguity,has_hierarchical_conflict,is_ambiguous
0,sha256_c4a98c965b13ba9993092f552104b9cbbb75ac9060714cf17ddd9abb08d2acec,AAAAMAGPK,toxic,no_information,positive,cytotoxic,True,False,False,False
1,sha256_c4a98c965b13ba9993092f552104b9cbbb75ac9060714cf17ddd9abb08d2acec,AAAAMAGPK,cytotoxic,no_information,positive,hemolytic,True,False,False,False
2,sha256_5958364b758eff160ee866452fd341c43eaf4faa36bb21e241da5354d3152b38,AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC,toxic,positive,positive,neurotoxic,False,False,False,False
3,sha256_2c475890e78b2fc16c34a3ad1c59cd9ab2faa21c9419eb916d1f6ef15ab94fbc,AAAISCVGSPECPPKCRAQGCKNGKCMNRKCECYYC,toxic,positive,positive,neurotoxic,False,False,False,False
4,sha256_ac92af50ce42fc29a5f93795b96e777e9382fc4562f01c4c0c8b1e521a4875df,AAAKAALNAVLVGANA,toxic,positive,positive,cytotoxic,False,False,False,False
5,sha256_ac92af50ce42fc29a5f93795b96e777e9382fc4562f01c4c0c8b1e521a4875df,AAAKAALNAVLVGANA,cytotoxic,positive,positive,hemolytic,False,False,False,False
6,sha256_49b50dbb3891c6777a005ec4ca33d00803e59a7adef0be7d1533f58b06aca837,AAAMSQVTN,toxic,no_information,positive,cytotoxic,True,False,False,False
7,sha256_49b50dbb3891c6777a005ec4ca33d00803e59a7adef0be7d1533f58b06aca837,AAAMSQVTN,cytotoxic,negative,positive,hemolytic,True,False,True,False
8,sha256_db2e2e9a4f62f220ca4eb7df064c48de8f03a71d677cfb694151ece0a4c6c088,AACEVAKNLNESLIDLQELGKYEQYIKW,toxic,negative,positive,cytotoxic,True,False,True,False
9,sha256_db2e2e9a4f62f220ca4eb7df064c48de8f03a71d677cfb694151ece0a4c6c088,AACEVAKNLNESLIDLQELGKYEQYIKW,cytotoxic,negative,positive,anti_mammalian_cells,True,False,True,False


In [12]:
if ambiguous_support is not None:
    print("Ambiguous-support table")
    display(ambiguous_support.head(20))
else:
    print("maomao_ambiguous_support.csv is not available.")


Ambiguous-support table


,id,sequence,cytotoxic,hemolytic,anti_mammalian_cells,neurotoxic,toxic
0,sha256_c70d828a62440a19acb4c85c0d1cff29a5b340ac5ba671e7d5869d2e754783db,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,999,30-40,999,999,70-80
1,sha256_339c8a3cc668390a015e03713838538b3d0b9cb06f5f83f2ee2a48ce4214ea55,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCET,999,50-60,999,999,999
2,sha256_153b1a5d1e21707009742f786481189da777d83065975c02d0b4e15b116d2d5a,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCRT,999,50-60,999,999,999
3,sha256_a5d113a92d6b2fa74edbfcdab58ec77c51bae0c4a9e0463dc562bf50b30b4ac5,AAAAGSCVWGAVNYTSDCAAECLLRGYKGGHCGSFANVNCWCET,999,50-60,999,999,999
4,sha256_4845a1ce228318f4638f14bab722a427702e3f033bd39d0c05571dbce895f969,AAAAGSCVWGAVNYTSDCAAECLLRGYKGGHCGSFANVNCWCRT,999,50-60,999,999,999
5,sha256_7244e99d35071fea873b29bd1486e44d8f54b24cebb36eb52124d908f29c1de3,AAAAGSCVWGAVNYTSDCNGECKRRGYKGGHCGSFANVNCWCET,999,50-60,999,999,999
6,sha256_6bde0915a269b5cd473273b55a021427903b3e56e0ac5e50e1f36830b70844d2,AAAAGSCVWGAVNYTSDCNGECKRRGYKGGHCGSFANVNCWCRT,999,50-60,999,999,999
7,sha256_966d40d882bc0c70c877f560e9d852f3bb428d473f776153641fd6f047430536,AAAAGSCVWGAVNYTSDCNGECLLRGYKGGHCGSFANVNCWCET,999,50-60,999,999,999
8,sha256_9be525d06c99aca7923dc92ea15e6562ad9b837b464f6134a661dd68b6d0e541,AAAAGSCVWGAVNYTSDCNGECLLRGYKGGHCGSFANVNCWCRT,999,50-60,999,999,999
9,sha256_5c003174dba016a6ab5c305d281365cc4d92ff129710e55128668e591e026b65,AAAAPGAAGGAQLPLGNRERKAGCKNFFWKTFSSC,999,999,999,999,10-20


##### 6. Basic resource queries

These examples operate directly on the frozen sequence-level pivot. They do not redefine the MAOMAO ontology or evidence states.


##### Example A — explicit positive neurotoxicity evidence


In [13]:
neuro_positive = maomao.loc[
    maomao["neurotoxic"].eq(1),
    ["id", "sequence", "sequence_length", *ENDPOINTS],
].copy()

print(f"Positive neurotoxic peptides: {len(neuro_positive):,}")
display(neuro_positive.head(15))


Positive neurotoxic peptides: 1,481


,id,sequence,sequence_length,toxic,cytotoxic,hemolytic,cytolysis,neurotoxic,embryotoxic,ichthyotoxic
36,sha256_5958364b758eff160ee866452fd341c43eaf4faa36bb21e241da5354d3152b38,AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC,35,1,999,999,999,1,999,999
37,sha256_2c475890e78b2fc16c34a3ad1c59cd9ab2faa21c9419eb916d1f6ef15ab94fbc,AAAISCVGSPECPPKCRAQGCKNGKCMNRKCECYYC,36,1,999,999,999,1,999,999
71,sha256_ed4da599cb41fed5e529543c21e2dde474e515a53d661d760f7b80cba01645c3,AACKCDDEGPDIRTAPLTGTVDLGSCNAGWEKCASYYTIIADCCRKKK,48,1,999,999,999,1,999,999
72,sha256_98dc6a87e4166aa098cb5a1f100297e7fdf5526f1b2bad74d9901bb8fd58ac79,AACLGMFESCDPNNDKCCPNRECNRKHKWCKYKLW,35,1,999,999,999,1,999,999
77,sha256_4fcb4182aabded96c474982c49c6f28be2320f9af6073c255bdc67bff476a631,AACYSSDCRVKCRAMGFSSGKCIDSKCKCYK,31,1,999,999,999,1,999,999
78,sha256_99bc769deaf2ac9091c0cc910f838928f94162341f49720eec48fb65b74a9101,AACYSSDCRVKCVAMGFSSGKCINSKCKCYK,31,1,999,999,999,1,999,999
243,sha256_877d478be9bd890d2d9bb00bcc2e9f1b0183fb33a92f1903074c48786d755142,AAKYCKLPVRYGPCKKKIPSFYYKWKAKQCLPFDYSGCGGNANRFKTIEECRRTCVG,57,1,999,999,999,1,999,999
348,sha256_71f68386219f39cae9ab5390709d1af57ad0d0509ec0239110c4697dd3cabb32,AAPCFCPGKPDRGDLWILRGTCPGGYGYTSNCYKWPNICCYPH,43,1,999,999,999,1,999,999
350,sha256_50a1534c0ec57dd750f4ce2dc4b63da9c590724be8dd6f572805f2b17547ec4b,AAPCFCSGKPGRGDLWILRGTCPGGYGYTSNCYKWPNICCYPH,43,1,0,0,999,1,999,999
352,sha256_a27c299c82f1b37534f4a7524e62843486668baf38aa890da7b63728256c3643,AAPCSCPGKPGRGDLWIFRGTCPGGYGYTSNCYKWPNICCYPH,43,1,999,999,999,1,999,999


##### Example B — explicit positive hemolytic evidence


In [14]:
hemolytic_positive = maomao.loc[
    maomao["hemolytic"].eq(1),
    ["id", "sequence", "sequence_length", *ENDPOINTS],
].copy()

print(f"Positive hemolytic peptides: {len(hemolytic_positive):,}")
display(hemolytic_positive.head(15))


Positive hemolytic peptides: 4,714


,id,sequence,sequence_length,toxic,cytotoxic,hemolytic,cytolysis,neurotoxic,embryotoxic,ichthyotoxic
21,sha256_c4a98c965b13ba9993092f552104b9cbbb75ac9060714cf17ddd9abb08d2acec,AAAAMAGPK,9,1,1,1,999,999,999,999
40,sha256_ac92af50ce42fc29a5f93795b96e777e9382fc4562f01c4c0c8b1e521a4875df,AAAKAALNAVLVGANA,16,1,1,1,999,999,999,999
48,sha256_49b50dbb3891c6777a005ec4ca33d00803e59a7adef0be7d1533f58b06aca837,AAAMSQVTN,9,1,1,1,999,999,999,999
73,sha256_0662c8bcd38afc071643454be9713491e796ba35ccf8c6c57f0fe41f7ce58ee1,AACSDRAHGHICESFKSFCKDSGRNGVKLRANCKKTCGLC,40,1,1,1,999,999,999,999
100,sha256_a4caf9006c1e1ba1df57205a3e328ca7118a0c62f9ec1d7f863b1f485baa3107,AAEEAARR,8,1,1,1,999,999,999,999
122,sha256_312b424bf13f39212519711cb697c9670a1228b8ae63fac6e6214d2b7a374cd5,AAGKVLKLLKKLL,13,1,1,1,999,999,999,999
124,sha256_537720593ca06e5b0eb7222df3fae0319e78d4e805e83a46059c2a7a5a81593c,AAGLAMLFLGILSAAGSTMGARA,23,2,2,1,999,999,999,999
166,sha256_764e0880810ecd41e890179353326f835c4f44277f7ac883d204d5fa0892f38a,AAIGSSKPK,9,1,1,1,999,999,999,999
208,sha256_721c39b1fa04c8fa868bdf83f2a6feb08533ea924bd392da0f6ba64718185762,AAKKLFKKILKYL,13,1,1,1,999,999,999,999
213,sha256_e2635c651187d6b7f9060145a859f0a0e4090abe8ff3f42247991445279658bb,AAKKVLKLLKKLL,13,1,1,1,999,999,999,999


##### Example C — a multi-endpoint query

Because endpoint states are retained as separate columns, a user can construct task-specific subsets directly.

The example below selects sequences that are positive for both `cytotoxic` and `hemolytic` and have an explicit neurotoxicity label (`0` or `1`).


In [15]:
multi_endpoint_subset = maomao.loc[
    maomao["cytotoxic"].eq(1)
    & maomao["hemolytic"].eq(1)
    & maomao["neurotoxic"].isin([0, 1]),
    ["id", "sequence", "sequence_length", *ENDPOINTS],
].copy()

print(f"Combined query returned {len(multi_endpoint_subset):,} peptides.")
display(multi_endpoint_subset.head(15))


Combined query returned 12 peptides.


,id,sequence,sequence_length,toxic,cytotoxic,hemolytic,cytolysis,neurotoxic,embryotoxic,ichthyotoxic
3090,sha256_528acd00a2dd703aef334db1464f5cb8f03b6ed1a09844ca27ade24da2fa4876,ANKRPIWIMGHMVNAIYQIDEFVNLGANSIETDVS,35,1,1,1,999,0,999,999
17236,sha256_f804478ae0deb3202be3fc3b7db68e9bfc2c062252d10cd9f0b3d2f26cef5534,GKLTDSQEDYIRHVWDDVNRKLITAKALERVNLVAEALSSNYH,43,2,1,1,999,0,999,999
20091,sha256_6cd2ff821afeaaf63809d60c15b19075595e8c73221f5df2338262d1ee45e21f,GTWDDIGQGIGRVAYWVGKAMGNMSDVNQASRINRKKKH,39,2,1,1,999,0,999,999
21485,sha256_ac7dcfbd624fdbd018ebac6fb5abd6dd09cc0be60e9abfaabae712b0752932c0,HLLQFGDLIDKIAGRSGFWYYGFYGCYCGLGGRGRPQDATDRCCFVHDCC,50,1,1,1,999,0,999,999
24714,sha256_927834f547127733a85b7d4f7eecc7ae1108bf6fd4b8581e8a379712a718716f,IWLTALKFLGKNLGKHLAKQQLAKL,25,2,1,1,1,0,999,999
28649,sha256_d960b9e9b4e78397ff193a186ca5a9b8d481a727ffd65bea875487b84422cc81,KSCCPTTTARNIYNTCRFGGGSRPVCAKLSGCKIISGTKCDSGWNH,46,2,1,1,999,0,999,999
41403,sha256_146eea7fd916cdb1e80bb763b0ab52e9ae84032e391bff6f0df1d54c4f032a7d,MKCLTKYSRVSETSQTCHVWQNLCFKKWQKGKKVSRGCTATCPKPKKDEVIQCCAKDKCNK,61,1,1,1,999,1,999,999
41722,sha256_e05808d2429f5164c838e6b5a3ef11906a3fd42d1f7b6c5294c269134f69eb66,MKILYLLFAFLFLAFLSEPGNAYKRCHKKGGHCFPKTVICLPPSSDFGKMDCRWKWKCCKKGSVNNAISI,70,1,1,1,999,1,999,999
43068,sha256_fd9dbfc9abd60a23ec710c883878326b90bc8061772121df61aa6f7509eccda3,MKTIVLLFVLALVFCTLEMGIVEAGFGCPFNQGQCHKHCQSIRRRGGYCDGFLKQRCVCYRK,62,1,1,1,999,1,999,999
63114,sha256_c1b80a739779919d00638f3d552dda1363be4e44363175745461fd9d0ec37321,SLGGKPDLRPCHPPCHYIPRPKPR,24,1,1,1,999,1,999,999


##### Example D — inspect ambiguous endpoint assignments


In [16]:
ambiguous_any = maomao.loc[
    maomao[ENDPOINTS].eq(2).any(axis=1),
    ["id", "sequence", "sequence_length", *ENDPOINTS],
].copy()

print(f"Peptides ambiguous for at least one endpoint: {len(ambiguous_any):,}")
display(ambiguous_any.head(15))


Peptides ambiguous for at least one endpoint: 13,177


,id,sequence,sequence_length,toxic,cytotoxic,hemolytic,cytolysis,neurotoxic,embryotoxic,ichthyotoxic
0,sha256_c70d828a62440a19acb4c85c0d1cff29a5b340ac5ba671e7d5869d2e754783db,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,33,2,0,2,999,999,999,999
9,sha256_339c8a3cc668390a015e03713838538b3d0b9cb06f5f83f2ee2a48ce4214ea55,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCET,44,0,0,2,999,999,999,999
10,sha256_153b1a5d1e21707009742f786481189da777d83065975c02d0b4e15b116d2d5a,AAAAGSCVWGAVNYTSDCAAECKRRGYKGGHCGSFANVNCWCRT,44,999,0,2,999,999,999,999
11,sha256_a5d113a92d6b2fa74edbfcdab58ec77c51bae0c4a9e0463dc562bf50b30b4ac5,AAAAGSCVWGAVNYTSDCAAECLLRGYKGGHCGSFANVNCWCET,44,0,0,2,999,999,999,999
12,sha256_4845a1ce228318f4638f14bab722a427702e3f033bd39d0c05571dbce895f969,AAAAGSCVWGAVNYTSDCAAECLLRGYKGGHCGSFANVNCWCRT,44,0,0,2,999,999,999,999
13,sha256_7244e99d35071fea873b29bd1486e44d8f54b24cebb36eb52124d908f29c1de3,AAAAGSCVWGAVNYTSDCNGECKRRGYKGGHCGSFANVNCWCET,44,0,0,2,999,999,999,999
14,sha256_6bde0915a269b5cd473273b55a021427903b3e56e0ac5e50e1f36830b70844d2,AAAAGSCVWGAVNYTSDCNGECKRRGYKGGHCGSFANVNCWCRT,44,0,0,2,999,999,999,999
15,sha256_966d40d882bc0c70c877f560e9d852f3bb428d473f776153641fd6f047430536,AAAAGSCVWGAVNYTSDCNGECLLRGYKGGHCGSFANVNCWCET,44,0,0,2,999,999,999,999
16,sha256_9be525d06c99aca7923dc92ea15e6562ad9b837b464f6134a661dd68b6d0e541,AAAAGSCVWGAVNYTSDCNGECLLRGYKGGHCGSFANVNCWCRT,44,999,0,2,999,999,999,999
22,sha256_5c003174dba016a6ab5c305d281365cc4d92ff129710e55128668e591e026b65,AAAAPGAAGGAQLPLGNRERKAGCKNFFWKTFSSC,35,2,999,999,999,999,999,999


##### 7. Retrieve and inspect one peptide

The stable MAOMAO `id` provides the common key for moving from the compact master resource to additional released assets.

The example below preferentially selects a positive neurotoxic peptide.


In [17]:
if len(neuro_positive):
    example_id = neuro_positive.iloc[0]["id"]
else:
    example_id = maomao.iloc[0]["id"]

entity = maomao.loc[maomao["id"].eq(example_id)].iloc[0]

print("id              :", example_id)
print("sequence length :", entity["sequence_length"])
print("sequence        :", entity["sequence"])
print("\nEndpoint states:")

for endpoint in ENDPOINTS:
    code_value = entity[endpoint]
    print(f" - {endpoint:<12}: {code_value} ({STATE_LABELS.get(code_value, 'Unknown')})")


id              : sha256_5958364b758eff160ee866452fd341c43eaf4faa36bb21e241da5354d3152b38
sequence length : 35
sequence        : AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC

Endpoint states:
 - toxic       : 1 (Positive)
 - cytotoxic   : 999 (No information)
 - hemolytic   : 999 (No information)
 - cytolysis   : 999 (No information)
 - neurotoxic  : 1 (Positive)
 - embryotoxic : 999 (No information)
 - ichthyotoxic: 999 (No information)


##### Ambiguity-support records for the selected peptide


In [18]:
def rows_for_entity(df, entity_id, sequence):
    if df is None or df.empty:
        return None

    if "id" in df.columns:
        subset = df.loc[df["id"].astype(str).eq(str(entity_id))].copy()
        if len(subset):
            return subset

    if "sequence" in df.columns:
        subset = df.loc[df["sequence"].astype(str).eq(str(sequence))].copy()
        if len(subset):
            return subset

    return df.iloc[0:0].copy()

selected_ambiguous = rows_for_entity(
    ambiguous_support,
    example_id,
    entity["sequence"],
)

if selected_ambiguous is None:
    print("Ambiguous-support table is not available.")
elif selected_ambiguous.empty:
    print("No ambiguity-support rows were found for this peptide.")
else:
    display(selected_ambiguous)


No ambiguity-support rows were found for this peptide.


##### Hierarchy-audit records for the selected peptide


In [19]:
selected_hierarchy = rows_for_entity(
    hierarchy_audit,
    example_id,
    entity["sequence"],
)

if selected_hierarchy is None:
    print("Hierarchy-audit table is not available.")
elif selected_hierarchy.empty:
    print("No hierarchy-change rows were found for this peptide.")
else:
    display(selected_hierarchy)


,id,sequence,endpoint,status_before_hierarchy,status,hierarchy_source,is_hierarchy_inferred,hierarchy_blocked_by_ambiguity,has_hierarchical_conflict,is_ambiguous
2,sha256_5958364b758eff160ee866452fd341c43eaf4faa36bb21e241da5354d3152b38,AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC,toxic,positive,positive,neurotoxic,False,False,False,False


##### Endpoint-level supporting context

The Core Layer can contain endpoint-specific integrated assets such as:

```text
processed_data/integrating_and_cleaning_data/<endpoint>/
├── positive.csv
├── negative.csv
├── ambiguous_data.csv
├── sequence_negative_evidece.csv
├── sequence_by_organism.csv
└── metadata.json
```

The following cell searches these released tables for the selected sequence without assuming that every endpoint has every file.


In [20]:
INTEGRATED_ROOT = PROCESSED_ROOT / "integrating_and_cleaning_data"

support_hits = []

if INTEGRATED_ROOT.is_dir():
    for endpoint_dir in sorted(p for p in INTEGRATED_ROOT.iterdir() if p.is_dir()):
        for filename in [
            "positive.csv",
            "negative.csv",
            "ambiguous_data.csv",
            "sequence_negative_evidece.csv",
            "sequence_by_organism.csv",
        ]:
            path = endpoint_dir / filename

            if not path.is_file():
                continue

            try:
                header = pd.read_csv(path, nrows=0)
            except Exception:
                continue

            match_column = None
            match_value = None

            if "id" in header.columns:
                match_column = "id"
                match_value = example_id
            elif "sequence" in header.columns:
                match_column = "sequence"
                match_value = entity["sequence"]

            if match_column is None:
                continue

            matched_chunks = []
            for chunk in pd.read_csv(path, chunksize=50000):
                matched = chunk.loc[
                    chunk[match_column].astype(str).eq(str(match_value))
                ]
                if len(matched):
                    matched_chunks.append(matched)

            if matched_chunks:
                hit = pd.concat(matched_chunks, ignore_index=True)
                support_hits.append((endpoint_dir.name, filename, hit))

if not support_hits:
    print("No endpoint-level supporting rows were found for the selected peptide.")
else:
    for endpoint_name, filename, hit in support_hits:
        print(f"\n{endpoint_name}/{filename} — {len(hit)} row(s)")
        display(hit.head(20))



neurotoxic/positive.csv — 1 row(s)


,sequence,NTXpred,NTXpred2,MultiTox,Peptipedia2.0,BiToxNet,counts_1,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
0,AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC,999,1,999,999,1,2,0,0,3,True,False,True,False,False,0.0,100.0



neurotoxic/sequence_by_organism.csv — 1 row(s)


,id,sequence,human,no information
0,sha256_5958364b758eff160ee866452fd341c43eaf4faa36bb21e241da5354d3152b38,AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC,1,1



toxic/positive.csv — 1 row(s)


,sequence,BIOPEP-UWM,CAPTP,CICERON,CSM-Toxin,HyPepTox-Fuse,iAMPCN,MultiPep,Pep-Lab_db,peptidereactor,PeptiTox,PLPTP,ProToxin,SATPdb,StrucToxNet,tAMPer,ToxDL 2.0,ToxGIN,ToxIBTL,ToxinPred,ToxinPred 2.0,ToxinPred 3.0,ToxiPep,ToxMSRC,ToxTeller,TPpred-LE,UniDL4BioPep,Zhao et al.,Peptipedia2.0,ProToxin_unlabel,counts_1,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
0,AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC,999,1,999,1,1,999,1,999,999,1,1,1,1,1,999,1,1,1,1,1,1,1,1,999,1,1,1,999,999,20,0,0,9,True,False,True,False,False,0.0,100.0



toxic/sequence_by_organism.csv — 1 row(s)


,id,sequence,animals,human,"human, animals",no information
0,sha256_5958364b758eff160ee866452fd341c43eaf4faa36bb21e241da5354d3152b38,AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC,999,1,1,1


##### 8. Discover released numerical representations

MAOMAO distributes precomputed numerical sequence representations so users do not need to regenerate PLM embeddings before downstream reuse.

The current release architecture supports several PLM families plus one-hot encoding.


In [21]:
REPRESENTATION_ROOT_CANDIDATES = [
    MAOMAO_ROOT / "numerical_representation_data" / "maomao",
    MAOMAO_ROOT / "embedding_layer" / "numerical_representation_data" / "maomao",
]

REPRESENTATION_ROOT = next(
    (path for path in REPRESENTATION_ROOT_CANDIDATES if path.is_dir()),
    None,
)

representation_rows = []

if REPRESENTATION_ROOT is None:
    print("No released numerical_representation_data/maomao directory was found.")
else:
    for path in sorted(REPRESENTATION_ROOT.rglob("full_data.csv")):
        header = pd.read_csv(path, nrows=0)
        p_features = [
            col for col in header.columns
            if re.fullmatch(r"p_\d+", str(col))
        ]

        representation_rows.append({
            "representation": path.parent.name,
            "family_directory": path.parent.parent.name,
            "feature_columns_p_contract": len(p_features),
            "total_columns": len(header.columns),
            "size_mb": round(path.stat().st_size / 1024**2, 2),
            "path": str(path),
        })

representation_inventory = pd.DataFrame(representation_rows)

if len(representation_inventory):
    display(representation_inventory)
else:
    print("No full_data.csv representation files were detected.")


,representation,family_directory,feature_columns_p_contract,total_columns,size_mb,path
0,ankh2_ext1,sylphy_embedding,1536,1546,1367.56,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/ankh2_ext1/full_data.csv
1,ankh3_large,sylphy_embedding,1536,1546,1306.21,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/ankh3_large/full_data.csv
2,esm2_t12_35M_UR50D,sylphy_embedding,480,490,396.47,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/esm2_t12_35M_UR50D/full_data.csv
3,esm2_t30_150M_UR50D,sylphy_embedding,640,650,528.54,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/esm2_t30_150M_UR50D/full_data.csv
4,esm2_t33_650M_UR50D,sylphy_embedding,1280,1290,1059.71,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/esm2_t33_650M_UR50D/full_data.csv
5,esm2_t6_8M_UR50D,sylphy_embedding,320,330,265.57,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/esm2_t6_8M_UR50D/full_data.csv
6,esmc_300m,sylphy_embedding,960,970,432.99,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/esmc_300m/full_data.csv
7,mistral_prot_v1_134M,sylphy_embedding,768,778,628.35,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/mistral_prot_v1_134M/full_data.csv
8,prot_bert,sylphy_embedding,1024,1034,870.37,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/prot_bert/full_data.csv
9,prot_t5_xl_uniref50,sylphy_embedding,1024,1034,856.48,/home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/prot_t5_xl_uniref50/full_data.csv


##### Inspect the selected peptide in one released PLM representation


In [22]:
PREFERRED_REPRESENTATIONS = [
    "esm2_t6_8M_UR50D",
    "ankh2_ext1",
    "esm2_t12_35M_UR50D",
    "esm2_t30_150M_UR50D",
]

selected_representation_path = None
selected_representation_name = None

if len(representation_inventory):
    for name in PREFERRED_REPRESENTATIONS:
        matches = representation_inventory.loc[
            representation_inventory["representation"].eq(name)
        ]
        if len(matches):
            selected_representation_path = Path(matches.iloc[0]["path"])
            selected_representation_name = name
            break

    if selected_representation_path is None:
        row = representation_inventory.iloc[0]
        selected_representation_path = Path(row["path"])
        selected_representation_name = row["representation"]

if selected_representation_path is None:
    print("No representation is available to inspect.")
else:
    header = pd.read_csv(selected_representation_path, nrows=0)
    p_features = sorted(
        [col for col in header.columns if re.fullmatch(r"p_\d+", str(col))],
        key=lambda col: int(col.split("_")[1]),
    )

    preview_features = p_features[:8]

    if not preview_features:
        excluded = {"id", "sequence", "label", "sequence_length", *ENDPOINTS}
        preview_features = [
            col for col in header.columns
            if col not in excluded
        ][:8]

    usecols = ["id", *preview_features]
    entity_representation = None

    for chunk in pd.read_csv(
        selected_representation_path,
        usecols=usecols,
        chunksize=50000,
    ):
        hit = chunk.loc[chunk["id"].astype(str).eq(str(example_id))]
        if len(hit):
            entity_representation = hit
            break

    print("Representation:", selected_representation_name)
    print("Path          :", selected_representation_path)
    print("Preview dims  :", preview_features)

    if entity_representation is None:
        print("The selected peptide was not found in this representation.")
    else:
        display(entity_representation)


Representation: esm2_t6_8M_UR50D
Path          : /home/nicole/Descargas/maomao/numerical_representation_data/maomao/sylphy_embedding/esm2_t6_8M_UR50D/full_data.csv
Preview dims  : ['p_0', 'p_1', 'p_2', 'p_3', 'p_4', 'p_5', 'p_6', 'p_7']


,id,p_0,p_1,p_2,p_3,p_4,p_5,p_6,p_7
36,sha256_5958364b758eff160ee866452fd341c43eaf4faa36bb21e241da5354d3152b38,-0.036908,0.137569,0.045098,0.269488,-0.066113,0.024182,-0.344842,-0.259612


##### 9. Discover endpoint-specific benchmark partitions

MAOMAO distributes compact `train.csv`, `val.csv`, and `test.csv` files containing only:

```text
id,label
```

Random and stratified partitions are representation-independent. Numerical features can therefore be attached later through the stable `id`.


In [23]:
SPLIT_ROOT_CANDIDATES = [
    MAOMAO_ROOT / "split_process",
    MAOMAO_ROOT / "benchmark_layer" / "split_process",
]

SPLIT_ROOT = next(
    (path for path in SPLIT_ROOT_CANDIDATES if path.is_dir()),
    None,
)

split_inventory_rows = []

if SPLIT_ROOT is None:
    print("No split_process directory was found.")
else:
    for train_path in sorted(SPLIT_ROOT.rglob("train.csv")):
        rel = train_path.relative_to(SPLIT_ROOT)
        parts = rel.parts

        dataset_name = parts[0] if len(parts) > 0 else None
        strategy = next(
            (p for p in parts if p in {"random_kfold", "stratified_kfold", "distance_kfold"}),
            None,
        )
        seed = next((p for p in parts if p.startswith("seed_")), None)
        fold = next((p for p in parts if p.startswith("fold_")), None)

        split_inventory_rows.append({
            "dataset": dataset_name,
            "strategy": strategy,
            "seed": seed,
            "fold": fold,
            "train_path": str(train_path),
        })

split_inventory = pd.DataFrame(split_inventory_rows)

if len(split_inventory):
    summary = (
        split_inventory
        .groupby(["dataset", "strategy"], dropna=False)
        .agg(
            seed_count=("seed", "nunique"),
            fold_count=("fold", "count"),
        )
        .reset_index()
    )
    display(summary)
else:
    print("No permanent train.csv split files were detected.")


,dataset,strategy,seed_count,fold_count
0,maomao_anti_mammalian_cells,random_kfold,30,150
1,maomao_anti_mammalian_cells,stratified_kfold,30,150
2,maomao_cytotoxic,random_kfold,30,150
3,maomao_cytotoxic,stratified_kfold,30,150
4,maomao_hemolytic,random_kfold,30,150
5,maomao_hemolytic,stratified_kfold,30,150
6,maomao_neurotoxic,random_kfold,30,150
7,maomao_neurotoxic,stratified_kfold,30,150
8,maomao_toxic,random_kfold,30,150
9,maomao_toxic,stratified_kfold,30,150


##### Inspect one released neurotoxicity partition


In [24]:
chosen_train_path = None

if len(split_inventory):
    preferred = split_inventory.loc[
        split_inventory["dataset"].astype(str).eq("maomao_neurotoxic")
        & split_inventory["strategy"].astype(str).eq("stratified_kfold")
    ].copy()

    if len(preferred):
        preferred["seed_preference"] = preferred["seed"].astype(str).eq("seed_113")
        preferred["fold_preference"] = preferred["fold"].astype(str).eq("fold_00")
        preferred = preferred.sort_values(
            ["seed_preference", "fold_preference"],
            ascending=False,
        )
        chosen_train_path = Path(preferred.iloc[0]["train_path"])
    else:
        chosen_train_path = Path(split_inventory.iloc[0]["train_path"])

if chosen_train_path is None:
    print("No released split is available to inspect.")
else:
    fold_dir = chosen_train_path.parent
    split_files = {
        name: fold_dir / f"{name}.csv"
        for name in ["train", "val", "test"]
    }

    released_splits = {
        name: pd.read_csv(path)
        for name, path in split_files.items()
        if path.is_file()
    }

    split_summary = pd.DataFrame([
        {
            "partition": name,
            "rows": len(df),
            "negative": int((df["label"] == 0).sum()) if "label" in df.columns else None,
            "positive": int((df["label"] == 1).sum()) if "label" in df.columns else None,
        }
        for name, df in released_splits.items()
    ])

    print("Selected fold:", fold_dir)
    display(split_summary)

    if "train" in released_splits:
        display(released_splits["train"].head(10))


Selected fold: /home/nicole/Descargas/maomao/split_process/maomao_neurotoxic/no_reduced/stratified_kfold/seed_113/fold_00


,partition,rows,negative,positive
0,train,2289,1219,1070
1,val,255,141,114
2,test,637,340,297


,id,label
0,sha256_456d4c07707b275e22614949f2666f9d6f89323af98789481f50d10b52acfbe5,0
1,sha256_160f5c4331eb1ae2ab68a7bb7b82e470f6d0b405dc1fd4a209d0392181c8181c,0
2,sha256_7ff8d4d1119af0eba9c20d8eb2578ecaa10738145bede075a744ed23ea3bca27,0
3,sha256_0e33dc8356f32882b72d07401fea1809974e833d9724de604c72d691cfd38972,0
4,sha256_a5438acb6bec4abf508cd1659e40d08e749f1ac6d85dd78257c1af9956957a4d,0
5,sha256_293ecd6c197688601316fa9387627818118e81cf4a57a62a5b1110d12690e076,1
6,sha256_f16af99291a057454bf833e7cdb17171bc0242ff9337d0360b337990f63f35d6,1
7,sha256_81a2818cc9e08f7ca969764422e8cbb5c1076224eaab33bac5657a9aa6264c26,1
8,sha256_63b5a81c51b1e2eb189228337de5678d31cb71ccca2aeae9efbf625639672231,1
9,sha256_119cad54b15edfa038b0dafde73a5b0ea7bbaa32ca246a6404e7e0b51d2a3c51,1


##### 10. Connect a fixed split to a released representation

This is the key interoperability step used by downstream machine-learning notebooks:

```text
released split (id, label)
          +
released representation (id, features)
          ↓
       join on id
          ↓
fixed benchmark membership + numerical features
```

For a lightweight getting-started demonstration, the cell below attaches only the first few representation dimensions. The classification notebook uses the complete feature matrix.


In [25]:
if (
    selected_representation_path is None
    or chosen_train_path is None
    or "train" not in locals().get("released_splits", {})
):
    print("A split and a numerical representation are both required for this example.")
else:
    split_train = released_splits["train"][["id", "label"]].copy()

    header = pd.read_csv(selected_representation_path, nrows=0)
    p_features = sorted(
        [col for col in header.columns if re.fullmatch(r"p_\d+", str(col))],
        key=lambda col: int(col.split("_")[1]),
    )

    preview_features = p_features[:8]

    if not preview_features:
        excluded = {"id", "sequence", "label", "sequence_length", *ENDPOINTS}
        preview_features = [
            col for col in header.columns
            if col not in excluded
        ][:8]

    representation_preview = pd.read_csv(
        selected_representation_path,
        usecols=["id", *preview_features],
    )

    reconstructed_preview = split_train.merge(
        representation_preview,
        on="id",
        how="inner",
        validate="one_to_one",
    )

    print("Split rows       :", len(split_train))
    print("Rows after join  :", len(reconstructed_preview))
    print("Representation   :", selected_representation_name)
    print("Preview features :", len(preview_features))

    if len(reconstructed_preview) != len(split_train):
        print(
            "WARNING: some split IDs were not represented in the selected numerical asset."
        )

    display(reconstructed_preview.head(10))


Split rows       : 2289
Rows after join  : 2289
Representation   : esm2_t6_8M_UR50D
Preview features : 8


,id,label,p_0,p_1,p_2,p_3,p_4,p_5,p_6,p_7
0,sha256_456d4c07707b275e22614949f2666f9d6f89323af98789481f50d10b52acfbe5,0,-0.004080,-0.066567,0.175647,0.209643,0.000847,-0.052775,-0.111190,-0.126991
1,sha256_160f5c4331eb1ae2ab68a7bb7b82e470f6d0b405dc1fd4a209d0392181c8181c,0,-0.063634,-0.366087,0.210686,0.280822,-0.000260,-0.186396,-0.154395,-0.020936
2,sha256_7ff8d4d1119af0eba9c20d8eb2578ecaa10738145bede075a744ed23ea3bca27,0,0.003285,-0.270395,0.129319,0.020003,0.039827,-0.204933,-0.131219,-0.027189
3,sha256_0e33dc8356f32882b72d07401fea1809974e833d9724de604c72d691cfd38972,0,0.094376,-0.270864,0.207918,0.206676,-0.126296,-0.323195,-0.139789,0.053446
4,sha256_a5438acb6bec4abf508cd1659e40d08e749f1ac6d85dd78257c1af9956957a4d,0,-0.079966,-0.100313,0.143695,0.008753,0.069350,-0.092436,-0.082348,-0.014184
5,sha256_293ecd6c197688601316fa9387627818118e81cf4a57a62a5b1110d12690e076,1,0.055179,0.006298,0.095180,0.130606,-0.088366,-0.053740,-0.362307,-0.278535
6,sha256_f16af99291a057454bf833e7cdb17171bc0242ff9337d0360b337990f63f35d6,1,0.068659,-0.154627,0.132273,0.218948,-0.002102,-0.023848,-0.315925,-0.217731
7,sha256_81a2818cc9e08f7ca969764422e8cbb5c1076224eaab33bac5657a9aa6264c26,1,0.051735,-0.268243,0.112528,0.174589,-0.113023,-0.197614,-0.265424,-0.115118
8,sha256_63b5a81c51b1e2eb189228337de5678d31cb71ccca2aeae9efbf625639672231,1,0.150906,-0.052103,0.080652,0.033463,0.048931,-0.026460,-0.244393,-0.183412
9,sha256_119cad54b15edfa038b0dafde73a5b0ea7bbaa32ca246a6404e7e0b51d2a3c51,1,-0.004247,0.004421,-0.037865,0.144498,-0.025720,-0.073224,-0.317943,-0.274526


##### 11. Optional: export a user-defined subset

The public MAOMAO release should be treated as read-only. Any user-defined derivative should be written outside the release.

The export below is disabled by default.


In [26]:
EXPORT_SUBSET = False

if EXPORT_SUBSET:
    output_dir = OUTPUT_ROOT / "00_getting_started"
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "positive_neurotoxic_peptides.tsv"
    neuro_positive.to_csv(output_path, sep="\t", index=False)

    print("Saved:", output_path.resolve())
else:
    print("Subset export disabled. Set EXPORT_SUBSET = True to write a derived table.")

Subset export disabled. Set EXPORT_SUBSET = True to write a derived table.
